# E1 multi-task grader — pre-training gates, then (separately) seeds 42 → 123 → 2026

**Model.** P unchanged (ImageNet ConvNeXt-Tiny → average pooling → LayerNorm → CORN) plus one 1×1 convolution on the final
16×16×768 feature map that predicts the frozen Stage-4 lesion maps (MA, HE, EX, SE) pooled to 16×16. 3,076 added parameters.
**Loss.** L = L_CORN + 1.0 · L_lesion (mean sigmoid cross-entropy against the soft teacher targets). λ = 1 is fixed.
**Protocol.** P's, unchanged: batch 2, AdamW 1e-4 / wd 0.05, weighted CORN, ≤ 50 epochs, val-QWK early stopping, mixed precision,
P's augmentation and epoch order, seeds 42 / 123 / 2026, BEST by validation QWK. No vessel supervision. No IDRiD.

**Cells 4–7 are the gates. They train nothing.**
- Cell 4 — Gate 3, targets: statistics and hard checks of all 2,921 training targets (label-free); gives the bias prior.
- Cell 5 — Gate 1, P parity: E1's grading path with each pinned P BEST checkpoint reproduces P on the 730 validation images
  (float32: 1e-4; mixed_float16 vs the stored P tables: logits 0.05, probabilities 0.01; grades identical), and equals P at initialisation.
- Cell 6 — Gate 2, log aliases: the actual Keras log keys of the two-output model on this runtime, and that the checkpoint
  state and the stored history receive `val_QWK`. One epoch on a 5-image synthetic bundle with random weights — mechanics only.
- Cell 7 — gradient check on one real batch: each loss alone reaches every encoder variable.

A failed gate stops here: no tolerance is loosened and nothing is trained.
**Cell 8 trains** and is switched off (`RUN_TRAINING = False`). It refuses to start unless the three gates are recorded as passed.

**Cell 3F holds one RUN flag per gate.** A gate set to False is not run again: its saved result on Drive is loaded, and is accepted only if it is an official PASS for this bundle and these P checkpoints. With the flags set, the notebook can be run top to bottom up to cell 7.

**Cell 9 is the pre-registered mechanism probe** (frozen P and frozen E1 encoders, the same fresh 1×1 probe on each); it trains no grader. Cell 8 trains only when `RUN_TRAINING` is True.

**Cell 10 is E2, the shuffled-target control** (E1 with each training image's lesion target replaced by its partner's in one fixed derangement, seed 20261001); it trains when `RUN_E2_TRAINING` is True. Cell 11 runs the same fresh probe on E2.

In [ ]:
# ==== [1] SETUP ====
import os, subprocess, sys
REPO_URL = "https://github.com/yasodharan27/diabetic_retinoplasty.git"
REPO_DIR = "/content/diabetic_retinoplasty"
BRANCH = "main"
if not os.path.isdir(os.path.join(REPO_DIR, ".git")):
    subprocess.run(["git", "clone", "--branch", BRANCH, REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(["git", "-C", REPO_DIR, "pull", "origin", BRANCH], check=True)
for _p in (REPO_DIR, os.path.join(REPO_DIR, "colab", "common")):
    if _p not in sys.path:
        sys.path.insert(0, _p)
REPO_COMMIT = subprocess.run(["git", "-C", REPO_DIR, "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip()
import setup as colab_setup
SETUP_INFO = colab_setup.setup()          # Drive, repo, requirements, env vars. No TJDR / IDRiD / Stage-4 weights needed.
import stage4_v2_setup                    # only for stage_bundle (remount-safe Drive -> local copy)
import glob, json, posixpath
import numpy as np, pandas as pd, tensorflow as tf
import colab_config
import arch1_data as ad, arch1_train as at
import e1_data as ed, e1_model as em, e1_train as et, e1_gates as eg
import multiseed_runs as msr, pl_convnext as pl, weighted_corn
import pipeline_v2_config as v2cfg, stage34_cache_v2 as cache
assert tf.config.list_physical_devices("GPU"), "GPU runtime required"
print("repo", REPO_COMMIT)

In [ ]:
# ==== [2] INPUTS: Stage-4 teacher identity, bundle, P protocol, pinned P BEST checkpoints ====
EXP = colab_config.DRIVE.experiments_root
DRIVE_ROOT = colab_config.DRIVE.project_root
EXPECTED_MODEL_SHA = "cb5fc7a8d370af7d2ae191cadaebdde858f71820d147fb76f852b757766f8ad8"   # gate-passed Stage-4 v2 (§47/§48)
GEN4 = cache.stage4_generation_id(EXPECTED_MODEL_SHA, len(v2cfg.STAGE4_V2A_CLASSES))
assert GEN4 == "s4v2-cb5fc7a8d370-K4", GEN4
BUNDLE_ID = cache.bundle_id(v2cfg.STAGE2_RGB_GENERATION, v2cfg.STAGE3_GENERATION, GEN4)
SIX_RUN_DIR = msr.experiment_root(posixpath.join(EXP, "ImprovedTraining"), "improved_multiseed_2026_09")
PREREG, _ = msr.load_and_verify_preregistration(posixpath.join(SIX_RUN_DIR, msr.PREREGISTRATION_FILENAME))
CLASS_WEIGHTS = list(PREREG["class_weights"])
at.assert_protocol(PREREG, CLASS_WEIGHTS)
SEEDS = et.SEEDS
assert SEEDS == (42, 123, 2026) and et.LAMBDA == 1.0 and at.EMA == "none"
assert em.LESION_CLASSES == ("MA", "HE", "EX", "SE") and em.ADDED_PARAMETERS == 3076 and ed.GRID == 16
# The pinned P BEST checkpoints (SHA-256) are the ones recorded in the frozen evaluation lock file (record §60).
_PINS = json.load(open(posixpath.join(REPO_DIR, "idrid_grading_protocol.json"), encoding="utf-8"))["checkpoints"]
P_PATHS = {s: posixpath.join(DRIVE_ROOT, _PINS[f"p_seed{s}"]["path"]) for s in SEEDS}
P_SHA256 = {s: _PINS[f"p_seed{s}"]["sha256"] for s in SEEDS}
PRETRAINED = pl.ensure_pretrained_weights(posixpath.join(EXP, "PL_ConvNeXtPriors", "pretrained_weights"))["local_path"]
assert cache.sha256_file(PRETRAINED) == pl.WEIGHTS_SHA256 == _PINS["convnext_pretrained"]["sha256"]
GATE_DIR = posixpath.join(EXP, et.EXPERIMENT, "gates")
RUN_DIRS = {s: et.run_dir_for(EXP, EXPECTED_MODEL_SHA, s) for s in SEEDS}
print("bundle", BUNDLE_ID, "| gates ->", GATE_DIR)
for _s in SEEDS:
    print(f"  seed {_s}: {et.seed_state(RUN_DIRS[_s]):<11} {RUN_DIRS[_s]} | P BEST {P_SHA256[_s][:12]}")

In [ ]:
# ==== [3] VERIFY THE BUNDLE ON DRIVE, STAGE THE RGB + LESION CACHES LOCALLY (no vessel files), VERIFY ====
DRIVE_ROOTS = dict(ad.DEFAULT_ROOTS)
LOCAL_ROOTS = {k: posixpath.join("/content/cache_v2_e1", os.path.basename(v)) for k, v in DRIVE_ROOTS.items()}
ad.Arch1Bundle(expected_bundle_id=BUNDLE_ID, expected_split_sha256=v2cfg.SPLIT_SHA256, expected_stage4_sha256=EXPECTED_MODEL_SHA)
ed.stage_inputs(BUNDLE_ID, DRIVE_ROOTS, LOCAL_ROOTS)
BUNDLE = ad.Arch1Bundle(expected_bundle_id=BUNDLE_ID, expected_split_sha256=v2cfg.SPLIT_SHA256,
                        expected_stage4_sha256=EXPECTED_MODEL_SHA, roots=LOCAL_ROOTS)
ed.require_inputs(BUNDLE, BUNDLE.population)
assert BUNDLE.stage4_sha256 == EXPECTED_MODEL_SHA and BUNDLE.stage4_generation == GEN4
_tr, _va, _split_sha = msr.verify_split()
_empty = set(v2cfg.APTOS_EMPTY_FOV_IDS)
assert _split_sha == v2cfg.SPLIT_SHA256
assert list(BUNDLE.train_ids) == [str(i) for i, _ in _tr if str(i) not in _empty] and len(BUNDLE.train_ids) == 2921
assert list(BUNDLE.val_ids) == [str(i) for i, _ in _va if str(i) not in _empty] and len(BUNDLE.val_ids) == 730
assert not os.path.isdir(BUNDLE.dirs["stage3_cache_v2"]), "the vessel cache must not be staged for E1"
print(json.dumps(BUNDLE.describe(), indent=1))

In [ ]:
# ==== [3F] WHICH GATES TO RUN IN THIS SESSION ====
# True  = run the gate now (its result is saved to GATE_DIR on Drive).
# False = do not run it; load the saved result. A saved result counts only if it is an OFFICIAL run that PASSED
#         for this bundle and these P checkpoints -- otherwise the cell stops and the gate must be run.
RUN_GATE3_TARGETS  = False     # cell 4  -- passed officially on 2026-10-05
RUN_GATE1_PARITY   = False     # cell 5  -- passed officially on 2026-10-05 (rerun after the fix of record 62.2)
RUN_GATE1_DIAGNOSTIC = False   # cell 5D -- diagnostic only, already run
RUN_GATE2_ALIAS    = False     # cell 6  -- passed officially on 2026-10-05
RUN_GRADIENT_CHECK = False     # cell 7  -- passed on 2026-10-05
RUN_TRAINING       = False     # cell 8  -- the three seeds finished on 2026-10-06 (record 63)
RUN_MECHANISM_PROBE = False    # cell 9  -- done on 2026-10-07 (record 63.2); its saved results are reused
RUN_E2_TRAINING    = True      # cell 10 -- E2, the shuffled-target control: THIS TRAINS three seeds (resumable)
RUN_E2_PROBE       = True      # cell 11 -- the same fresh frozen-encoder probe on E2 BEST (runs after cell 10)

def load_saved_gate(name, filename, check=None):
    _path = posixpath.join(GATE_DIR, filename)
    assert os.path.exists(_path), f"{name}: no saved result at {_path} -- set its RUN flag to True"
    _g = json.load(open(_path))
    assert _g.get("official") and _g.get("PASS"), (f"{name}: the saved result is not an official PASS "
                                                   f"({_g.get('kind')}, PASS={_g.get('PASS')}, {_g.get('failures')}) -- run the gate")
    if check is not None:
        check(_g)
    print(f"{name}: loaded saved official PASS from {_g.get('timestamp_utc')}")
    return _g
print({k: v for k, v in globals().items() if k.startswith("RUN_")})
# Latest code, also when this cell is rerun in a session that was started before a push.
import importlib
subprocess.run(["git", "-C", REPO_DIR, "pull", "origin", BRANCH], check=True)
import e1_probe as ep, e2_control as e2
for _m in (em, ed, et, eg, ep, e2):
    importlib.reload(_m)
print("repo", subprocess.run(["git", "-C", REPO_DIR, "rev-parse", "--short", "HEAD"], capture_output=True, text=True).stdout.strip())

In [ ]:
# ==== [4] GATE 3 -- TARGETS: all 2,921 training targets (label-free). Trains nothing. ====
if RUN_GATE3_TARGETS:
    GATE_TARGETS = eg.targets(BUNDLE, out_dir=GATE_DIR)
    print(json.dumps(GATE_TARGETS["statistics"], indent=1))
    assert GATE_TARGETS["PASS"], f"TARGET GATE FAILED -- stop. {GATE_TARGETS['failures']}"
else:
    def _same_bundle(g):
        assert g["bundle_fingerprint"] == BUNDLE.fingerprint and g["stage4_sha256"] == BUNDLE.stage4_sha256, "saved for another bundle"
    GATE_TARGETS = load_saved_gate("Gate 3 (targets)", "gate3_targets.json", _same_bundle)
LESION_PRIOR = GATE_TARGETS["lesion_prior"]
print("lesion prior (MA, HE, EX, SE):", [round(p, 5) for p in LESION_PRIOR],
      "| bias:", [round(b, 4) for b in GATE_TARGETS["lesion_prior_logits"]])

In [ ]:
# ==== [5] GATE 1 -- P PARITY on the 730 validation images, both tiers. Trains nothing. ====
if RUN_GATE1_PARITY:
    GATE_PARITY = eg.p_parity(BUNDLE, P_PATHS, PRETRAINED, LESION_PRIOR, policies=("float32", "mixed_float16"),
                              stored_root=EXP, expected_sha256=P_SHA256, out_dir=GATE_DIR)
    print(json.dumps({"PASS": GATE_PARITY["PASS"], "failures": GATE_PARITY["failures"],
                      "initialisation": GATE_PARITY["initialisation"],
                      "float32": {s: r["e1_vs_live_p"] for s, r in GATE_PARITY["tiers"]["float32"].items()},
                      "mixed_float16 E1 vs P (same runtime)": {s: r["e1_vs_live_p"] for s, r in GATE_PARITY["tiers"]["mixed_float16"].items()},
                      "mixed_float16 E1 vs stored P": {s: r.get("e1_vs_stored_p_best") for s, r in GATE_PARITY["tiers"]["mixed_float16"].items()},
                      "precision actually used": {p: {s: r["grading_output_dtype"] for s, r in t.items()} for p, t in GATE_PARITY["tiers"].items()},
                      "environment": GATE_PARITY["environment"]}, indent=1, default=float))
    assert GATE_PARITY["PASS"], "P PARITY FAILED -- stop. Do not loosen tolerances. Do not train."
else:
    def _same_checkpoints(g):
        assert {int(s): h for s, h in g["p_checkpoint_sha256"].items()} == P_SHA256, "saved for other P checkpoints"
        _used = {s: r.get("grading_output_dtype", {}).get("e1_grading") for s, r in g["tiers"]["mixed_float16"].items()}
        assert set(_used.values()) == {"float16"}, f"the saved mixed tier did not run in mixed precision: {_used} -- rerun the gate"
    GATE_PARITY = load_saved_gate("Gate 1 (P parity)", "gate1_p_parity.json", _same_checkpoints)

In [ ]:
# ==== [5D] GATE 1 DIAGNOSTIC -- no pass / fail, no rule or tolerance changed. Trains nothing. ====
# P recomputed under mixed_float16 on this runtime vs P's STORED validation logits, per image; and E1 vs that P.
if RUN_GATE1_DIAGNOSTIC:
    import importlib
    subprocess.run(["git", "-C", REPO_DIR, "pull", "origin", BRANCH], check=True)
    importlib.reload(eg)
    DIAG = eg.p_parity_diagnostic(BUNDLE, P_PATHS, PRETRAINED, LESION_PRIOR, EXP, expected_sha256=P_SHA256, out_dir=GATE_DIR)
    for _s, _r in DIAG["seeds"].items():
        _d = _r["recomputed_p_vs_stored_p"]
        print(f"\n===== seed {_s} | stored logit range {_r['stored_logit_range']} | stored table consistent {_r['stored_table_internally_consistent']}")
        print("logit |diff| (all 730 x 4):", _d["logit_abs_diff"])
        print("logit |diff| per-image max:", _d["logit_abs_diff_per_image_max"])
        print("probability |diff|:", _d["probability_abs_diff"])
        print("decoded grades differ:", _d["decoded_grades_differ"])
        print("worst image:", json.dumps(_d["worst_image"], indent=1))
        print("saturation:", json.dumps(_d["saturation"], indent=1))
        print("shift:", _d["shift"])
        print("E1 vs recomputed P (same runtime):", _r["e1_vs_recomputed_p_same_runtime"])
        print("P at batch 2 vs batch 8 (same policy):", _r["recomputed_p_other_batch_size_vs_this_one_same_policy"])
        print("P float32 vs stored:", _r["recomputed_p_float32_vs_stored_p"])
        print("P mixed vs P float32:", _r["recomputed_p_policy_vs_recomputed_p_float32"])
    print("\nenvironment:", DIAG["environment"], "| files ->", GATE_DIR)
else:
    print("diagnostic not run (RUN_GATE1_DIAGNOSTIC = False); saved result:", posixpath.join(GATE_DIR, "gate1_diagnostic.json"))

In [ ]:
# ==== [6] GATE 2 -- LOG ALIASES on this runtime's Keras. Synthetic 5-image bundle, random weights, ONE epoch: mechanics only. ====
if RUN_GATE2_ALIAS:
    import shutil, keras
    sys.path.insert(0, posixpath.join(REPO_DIR, "tests"))
    import v2_bundle_fixture as fx
    _work = "/content/e1_alias_gate"
    shutil.rmtree(_work, ignore_errors=True)
    _fx = fx.build(posixpath.join(_work, "data"))
    _fx_bundle = ad.Arch1Bundle(expected_bundle_id=_fx["bundle"]["bundle_id"], expected_split_sha256=v2cfg.SPLIT_SHA256,
                                expected_stage4_sha256=fx.MODEL_SHA, roots=_fx["roots"], expected_population=5)
    keras.mixed_precision.set_global_policy("float32")
    keras.utils.set_random_seed(7)
    _ref = keras.applications.ConvNeXtTiny(include_top=False, weights=None, include_preprocessing=True, pooling="avg",
                                           input_shape=(512, 512, 3), name=pl.BACKBONE_NAME)       # RANDOM weights
    GATE_ALIAS = eg.log_alias(_work, _fx_bundle, dict(fx.TRAIN + fx.VAL), _ref, CLASS_WEIGHTS, mixed_precision=True,
                              repo_dir=REPO_DIR, out_dir=GATE_DIR)
    print("Keras", keras.__version__, "| raw log keys:", GATE_ALIAS["raw_keras_log_keys"])
    print("after the alias callback:", GATE_ALIAS["keys_after_alias"])
    print("callback order:", GATE_ALIAS["callback_order"])
    print("stored history row:", GATE_ALIAS["stored_history_row"])
    assert GATE_ALIAS["PASS"], f"LOG ALIAS GATE FAILED -- stop. Do not train. {GATE_ALIAS['failures']}"
    del _ref
else:
    GATE_ALIAS = load_saved_gate("Gate 2 (log aliases)", "gate2_log_alias.json")
    print("raw Keras log keys:", GATE_ALIAS["raw_keras_log_keys"], "| Keras", GATE_ALIAS["environment"]["keras"])

In [ ]:
# ==== [7] GRADIENT CHECK on one real training batch (ImageNet-initialised E1, the run's precision). Trains nothing. ====
if RUN_GRADIENT_CHECK:
    tf.keras.backend.clear_session()
    _reference, _ = pl.load_reference(PRETRAINED)
    _model = et.build_compiled_model(SEEDS[0], LESION_PRIOR, _reference, CLASS_WEIGHTS)
    _grade = {str(i): int(g) for i, g in list(_tr) + list(_va)}
    _x, (_g, _t) = ed.make_epoch_sequence(BUNDLE, [(i, _grade[i]) for i in BUNDLE.train_ids[:2]], 0, SEEDS[0], 2, augment=True)[0]
    GATE_GRADIENT = eg.gradient_report(_model, _x["rgb"], _g, _t, CLASS_WEIGHTS)
    print(json.dumps(GATE_GRADIENT, indent=1), "\nparameters:", em.parameter_report(_model))
    with open(posixpath.join(GATE_DIR, "gradient_check.json"), "w") as fh:
        json.dump(GATE_GRADIENT, fh, indent=1)
    del _model, _reference
else:
    _path = posixpath.join(GATE_DIR, "gradient_check.json")
    assert os.path.exists(_path), "no saved gradient check -- set RUN_GRADIENT_CHECK = True"
    GATE_GRADIENT = json.load(open(_path))
    print("gradient check: loaded saved result")
assert GATE_GRADIENT["PASS"], f"GRADIENT CHECK FAILED -- stop. {GATE_GRADIENT['failures']}"
GATES = {"p_parity": GATE_PARITY, "log_alias": GATE_ALIAS, "targets": GATE_TARGETS}
print("\nGATES:", {k: ("PASS" if v["PASS"] and v["official"] else "FAIL") for k, v in GATES.items()}, "| gradient check:",
      "PASS" if GATE_GRADIENT["PASS"] else "FAIL")

In [ ]:
# ==== [8] TRAIN + EVALUATE SEEDS 42 -> 123 -> 2026 (resumable) -- THIS TRAINS when RUN_TRAINING is True ====
if RUN_TRAINING:
    assert all(g["PASS"] and g["official"] for g in GATES.values()) and GATE_GRADIENT["PASS"], "a gate has not passed"
    tf.keras.backend.clear_session()
    REFERENCE, _ = pl.load_reference(PRETRAINED)
    RESULTS = et.run_sequence(EXP, BUNDLE, LESION_PRIOR, REFERENCE, CLASS_WEIGHTS, repo_dir=REPO_DIR,
                              staging_root="/content/e1_staging",
                              gates={k: {"PASS": v["PASS"] and v["official"], "timestamp_utc": v["timestamp_utc"]} for k, v in GATES.items()})
else:
    print("training not run (RUN_TRAINING = False):", {s: et.seed_state(RUN_DIRS[s]) for s in SEEDS})

In [ ]:
# ==== [9] MECHANISM PROBE (record 62) -- frozen P and frozen E1 encoders, one fresh probe each. Trains NO grader. ====
# Per seed: the final 16x16x768 feature map of P BEST and of E1 BEST is computed once (encoders frozen), and the SAME fresh
# 1x1 probe (Adam 1e-3, batch 16, 5 epochs, no augmentation, final epoch) is trained on each to predict the E1 lesion targets.
# E1's trained lesion head and both grading heads are not used. The paired bootstrap is done afterwards from the saved logits.
if RUN_MECHANISM_PROBE:
    assert all(et.seed_state(RUN_DIRS[s]) == "complete" for s in SEEDS), "the three E1 runs must be complete"
    PROBE_DIR = posixpath.join(EXP, et.EXPERIMENT, "mechanism_probe")
    PROBE = ep.run(BUNDLE, P_PATHS, RUN_DIRS, PRETRAINED, LESION_PRIOR, PROBE_DIR, p_sha256=P_SHA256)
    print("\nprobe protocol:", ep.PROBE)
    for _s, _r in PROBE["seeds"].items():
        print(f"\n===== seed {_s}")
        for _k in ("p", "e1"):
            _f = _r[_k]["final"]
            print(f"  {_k:>2}: val cell AUROC", {c: round(v, 4) for c, v in _f["val_scores"].items()},
                  "| train", round(_f["train_scores"]["mean"], 4), "| val loss", round(_f["val_loss"], 4),
                  "| train loss", round(_f["train_loss"], 4))
            print("      per epoch (val loss, val mean AUROC):", [(c["epoch"], round(c["val_loss"], 4), round(c["val_mean_cell_auroc"], 4)) for c in _r[_k]["curve"]])
        print("  probe(E1) - probe(P):", {c: round(v, 4) for c, v in _r["difference_e1_minus_p"].items()})
    print("\nsaved ->", PROBE_DIR)
else:
    print("mechanism probe not run (RUN_MECHANISM_PROBE = False)")

In [ ]:
# ==== [10] E2 -- SHUFFLED-TARGET CONTROL, seeds 42 -> 123 -> 2026 (resumable). THIS TRAINS when RUN_E2_TRAINING is True ====
# E1 unchanged in every respect except the TRAINING lesion targets: each image gets the cached Stage-4 map of its partner in
# ONE fixed derangement of the training set (seed 20261001, the same for all three model seeds). Validation targets stay aligned.
# A control: not selected against E1, replaces nothing, tunes nothing. BEST by validation QWK, as E1.
E2_RUN_DIRS = {s: e2.run_dir_for(EXP, EXPECTED_MODEL_SHA, s) for s in SEEDS}
E2_PARTNER = e2.derangement(BUNDLE.train_ids)
E2_DERANGEMENT = e2.verify_derangement(E2_PARTNER, BUNDLE.train_ids, BUNDLE.val_ids)
assert e2.DERANGEMENT_SEED == 20261001 and E2_DERANGEMENT["fixed_points"] == 0 and E2_DERANGEMENT["images"] == 2921
assert all(E2_PARTNER[i] != i for i in BUNDLE.train_ids) and sorted(E2_PARTNER.values()) == sorted(BUNDLE.train_ids)
assert not set(E2_PARTNER.values()) & set(BUNDLE.val_ids)
print("derangement:", E2_DERANGEMENT)
if RUN_E2_TRAINING:
    assert all(g["PASS"] and g["official"] for g in GATES.values()) and GATE_GRADIENT["PASS"], "a gate has not passed"
    assert all(et.seed_state(RUN_DIRS[s]) == "complete" for s in SEEDS), "E1 must be complete (E2 is its control)"
    tf.keras.backend.clear_session()
    REFERENCE, _ = pl.load_reference(PRETRAINED)
    E2_RESULTS = e2.run_sequence(EXP, BUNDLE, LESION_PRIOR, REFERENCE, CLASS_WEIGHTS, repo_dir=REPO_DIR,
                                 staging_root="/content/e2_staging",
                                 gates={k: {"PASS": v["PASS"] and v["official"], "timestamp_utc": v["timestamp_utc"]} for k, v in GATES.items()})
    del REFERENCE
else:
    print("E2 training not run (RUN_E2_TRAINING = False):", {s: e2.seed_state(E2_RUN_DIRS[s]) for s in SEEDS})

In [ ]:
# ==== [11] E2 PROBE -- the SAME fresh frozen-encoder probe as record 63.2, on each E2 BEST encoder. Trains NO grader. ====
# Same probe form, optimiser, five epochs, seed, batch order and aligned targets; E2's trained lesion head is not used.
# The P and E1 probe results of record 63.2 are reused as saved; the comparison and bootstrap are done afterwards.
if RUN_E2_PROBE:
    assert all(e2.seed_state(E2_RUN_DIRS[s]) == "complete" for s in SEEDS), "the three E2 runs must be complete"
    E1_PROBE_DIR = posixpath.join(EXP, et.EXPERIMENT, "mechanism_probe")
    E2_PROBE_DIR = posixpath.join(EXP, e2.EXPERIMENT, "mechanism_probe")
    E2_PROBE = e2.run_probe(BUNDLE, E2_RUN_DIRS, PRETRAINED, LESION_PRIOR, E2_PROBE_DIR, E1_PROBE_DIR)
    _e1 = json.load(open(posixpath.join(E1_PROBE_DIR, "summary.json")))["seeds"]
    for _s in SEEDS:
        _r = E2_PROBE["seeds"][_s]["e2"]
        print(f"\n===== seed {_s}")
        for _k, _f in (("p", _e1[str(_s)]["p"]["final"]), ("e1", _e1[str(_s)]["e1"]["final"]), ("e2", _r["final"])):
            print(f"  {_k:>2}: val cell AUROC", {c: round(v, 4) for c, v in _f["val_scores"].items()},
                  "| train", round(_f["train_scores"]["mean"], 4), "| val loss", round(_f["val_loss"], 4))
        print("  E2 per epoch (val loss, val mean AUROC):", [(c["epoch"], round(c["val_loss"], 4), round(c["val_mean_cell_auroc"], 4)) for c in _r["curve"]])
        with open(posixpath.join(E2_RUN_DIRS[_s], "result.json")) as fh:
            _b = json.load(fh)["best"]
        print("  E2 grading: QWK", round(_b["metrics"]["qwk"], 4), "| best epoch index", _b["checkpoint"]["best_epoch"],
              "| trained head vs TRUE targets, mean cell AUROC", round(_b["lesion_head_validation"]["mean_cell_auroc"], 4))
    print("\nsaved ->", E2_PROBE_DIR)
else:
    print("E2 probe not run (RUN_E2_PROBE = False)")